# 05 · Find the satellite evidence

[Run this notebook in your browser ↗](https://jltobias.github.io/JupyterLite-Global-Fishing-Watch-and-Dark-Vessels/lab/index.html?path=05_satellite_catalog.ipynb)

**Evidence: real STAC metadata, with optional live discovery.** Learn to find acquisitions, inspect sensor properties, and separate a catalog record from actual pixel analysis. The core exercise uses three bundled Sentinel-1 metadata records for June 2024 near the southern Baltic.

Sources: [Microsoft Planetary Computer STAC documentation](https://planetarycomputer.microsoft.com/docs/quickstarts/reading-stac/) and [Sentinel-1 RTC catalog](https://planetarycomputer.microsoft.com/dataset/sentinel-1-rtc). Metadata is not an ICEYE product.

Run the setup cell first. Browser Python downloads numerical packages on first use. Core exercises use bundled files; live access is optional. Export your notebook to preserve work.

In [1]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["numpy", "pandas", "matplotlib", "scipy", "plotly"])
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import oceanlab as lab
from IPython.display import display
print("Runtime:", sys.version.split()[0], "| NumPy:", np.__version__, "| pandas:", pd.__version__)

Runtime: 3.13.14 | NumPy: 2.3.3 | pandas: 2.3.2


In [2]:
snapshot=json.loads((lab.DATA/"sentinel1_stac_snapshot.json").read_text())
def scene_table(collection):
    rows=[]
    for item in collection.get("features",[]):
        p=item["properties"]
        rows.append({"id":item["id"],"time_utc":p.get("datetime"),"orbit":p.get("sat:orbit_state"),
            "polarizations":str(p.get("sar:polarizations")),"mode":p.get("sar:instrument_mode"),
            "assets":", ".join(item["assets"])})
    return pd.DataFrame(rows)
display(scene_table(snapshot))
first=snapshot["features"][0]
display(pd.Series(first["assets"]["vv"]))

,id,time_utc,orbit,polarizations,mode,assets
0,S1A_IW_GRDH_1SDV_20240629T170134_20240629T1701...,2024-06-29T17:01:47.419630Z,ascending,"['VV', 'VH']",IW,"vh, vv, tilejson, rendered_preview"
1,S1A_IW_GRDH_1SDV_20240629T170109_20240629T1701...,2024-06-29T17:01:22.420072Z,ascending,"['VV', 'VH']",IW,"vh, vv, tilejson, rendered_preview"
2,S1A_IW_GRDH_1SDV_20240626T052527_20240626T0525...,2024-06-26T05:25:39.804357Z,descending,"['VV', 'VH']",IW,"vh, vv, tilejson, rendered_preview"


href             https://sentinel1euwestrtc.blob.core.windows.n...
file:size                                               1837390208
file:checksum                     1ae98cd3bbf19b429635daf7df985919
type             image/tiff; application=geotiff; profile=cloud...
roles                                                       [data]
title                      VV: vertical transmit, vertical receive
description      Terrain-corrected gamma naught values of signa...
raster:bands     [{'nodata': -32768, 'data_type': 'float32', 's...
dtype: object

## Optional: repeat the discovery query
Leave `LIVE = False` to reproduce the saved exercise. Enable it to query the public catalog without credentials. The request is limited to three records. Browser cross-origin restrictions, provider outages, or throttling can prevent live access; the message below identifies a failed query and keeps the saved snapshot explicitly labeled.

In [3]:
LIVE=False
query={"collections":["sentinel-1-rtc"],"bbox":[11.5,54.0,13.5,55.0],
       "datetime":"2024-06-01T00:00:00Z/2024-06-30T23:59:59Z","limit":3}
endpoint="https://planetarycomputer.microsoft.com/api/stac/v1/search"
if LIVE:
    try:
        if sys.platform=="emscripten":
            from pyodide.http import pyfetch
            response=await pyfetch(endpoint,method="POST",headers={"Content-Type":"application/json"},body=json.dumps(query))
            if not response.ok: raise RuntimeError(f"HTTP {response.status}")
            live=await response.json()
        else:
            import urllib.request
            request=urllib.request.Request(endpoint,data=json.dumps(query).encode(),headers={"Content-Type":"application/json"})
            with urllib.request.urlopen(request,timeout=30) as response: live=json.load(response)
        print("LIVE RESPONSE; empty results mean no records returned, not no vessels.")
        display(scene_table(live))
    except Exception as error:
        print("Live discovery failed:",type(error).__name__,"— use the labeled saved snapshot above.")
else:
    print("SAVED SNAPSHOT MODE — live network request disabled")

SAVED SNAPSHOT MODE — live network request disabled


## Design a multisensor comparison
Do not add unlike sensors as if their detections were independent counts of unique boats. Match acquisition time, footprint, resolution, viewing geometry and quality flags. A vessel can appear repeatedly across passes. Optical cloud screening has no direct radar equivalent; radar has its own sea-state and geometry limitations.

In [4]:
sensors=pd.DataFrame([
  ["Sentinel-1 SAR","Active microwave backscatter","Day/night, cloud tolerant","Speckle, sea state, geometry; not all small vessels"],
  ["Sentinel-2 optical","Reflected sunlight","Shape, wakes, coastal context","Cloud, sun, spatial resolution"],
  ["VIIRS night lights","Low-light radiance","Some brightly lit fleets","Moon/cloud screening, unlit vessels"],
  ["SST products","Surface temperature estimate","Environmental context","Cloud/gap-filling, skin vs bulk; no pathogen diagnosis"],
  ["Ocean color","Optical water properties","Phytoplankton context","Cloud, turbidity, retrieval error; not toxin identity"]
],columns=["source","measurement","strength","limitation"])
display(sensors)

,source,measurement,strength,limitation
0,Sentinel-1 SAR,Active microwave backscatter,"Day/night, cloud tolerant","Speckle, sea state, geometry; not all small ve..."
1,Sentinel-2 optical,Reflected sunlight,"Shape, wakes, coastal context","Cloud, sun, spatial resolution"
2,VIIRS night lights,Low-light radiance,Some brightly lit fleets,"Moon/cloud screening, unlit vessels"
3,SST products,Surface temperature estimate,Environmental context,"Cloud/gap-filling, skin vs bulk; no pathogen d..."
4,Ocean color,Optical water properties,Phytoplankton context,"Cloud, turbidity, retrieval error; not toxin i..."


## From catalog to calibrated crop
Use the repository's optional local `fetch_sar_window.py` after installing `requirements-eo.txt`. The script requests a fresh short-lived signature, reads a bounded window, and saves unsigned provenance. Full-resolution geospatial processing needs libraries and resources beyond the core browser exercises.

**Try it:** search a short interval, record the exact item IDs, and inspect polarization, orbit and mode before comparing images. Give one reason that an empty catalog result cannot establish an empty sea.